## Data Modelling

This notebook is used to create dimension tables and fact tables.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the four cleaned datasets

presentations_clean = pd.read_csv(
    INTERIM_DIR / "presentations_clean.csv")

seen_on_time_clean = pd.read_csv(
    INTERIM_DIR / "patients_seen_on_time_clean.csv")

within_4hrs_clean = pd.read_csv(
    INTERIM_DIR / "time_in_ed_within_4hrs_clean.csv")

time_in_ed_clean = pd.read_csv(
    INTERIM_DIR / "time_in_ed_clean.csv")

In [4]:
print("Presentations:", presentations_clean.shape)
print("Seen on time:", seen_on_time_clean.shape)
print("Within 4 hours:", within_4hrs_clean.shape)
print("Time in ED:", time_in_ed_clean.shape)

Presentations: (22678, 7)
Seen on time: (21972, 13)
Within 4 hours: (36289, 13)
Time in ED: (13611, 19)


### Create Dimension Tables

In [5]:
# Create reporting_unit_dim
reporting_unit_dim = pd.concat([
    presentations_clean[["reporting_unit", "reporting_unit_type", "state"]],
    seen_on_time_clean[["reporting_unit", "reporting_unit_type", "state"]],
    within_4hrs_clean[["reporting_unit", "reporting_unit_type", "state"]],
    time_in_ed_clean[["reporting_unit", "reporting_unit_type", "state"]]])

In [6]:
# Remove duplicate combinations
reporting_unit_dim = (reporting_unit_dim.drop_duplicates().sort_values(["reporting_unit_type", "state", "reporting_unit"]).reset_index(drop=True))



In [7]:
# Add surrogate keys
reporting_unit_dim.insert(0, "reporting_unit_key", range(1, len(reporting_unit_dim) + 1))

In [8]:
# Check the data
reporting_unit_dim.head(10)

,reporting_unit_key,reporting_unit,reporting_unit_type,state
0,1,North Canberra Hospital,Hospital,ACT
1,2,North Canberra Hospital (previously known as C...,Hospital,ACT
2,3,The Canberra Hospital,Hospital,ACT
3,4,Albury Wodonga Health [Albury Campus],Hospital,NSW
4,5,Armidale Hospital,Hospital,NSW
5,6,Auburn Hospital,Hospital,NSW
6,7,Ballina District Hospital,Hospital,NSW
7,8,Balranald Multi Purpose Service,Hospital,NSW
8,9,Bankstown Lidcombe Hospital,Hospital,NSW
9,10,Baradine Multi Purpose Service,Hospital,NSW


In [9]:
print(len(reporting_unit_dim))

402


In [10]:
# Check the uniqueness of the surrogate key

assert reporting_unit_dim["reporting_unit_key"].is_unique

In [ ]:
# Validate the natural identifier

assert (reporting_unit_dim.duplicated(subset=["reporting_unit", "reporting_unit_type"]).sum() == 0)

In [13]:
# Check state
assert reporting_unit_dim["state"].notna().all()

In [14]:
reporting_unit_dim["reporting_unit_type"].value_counts()

reporting_unit_type
Hospital                  311
Local Hospital Network     82
State                       8
National                    1
Name: count, dtype: int64

In [15]:
# Create financial_year_dim table

financial_year_dim = pd.concat([
    presentations_clean[["year"]],
    seen_on_time_clean[["year"]],
    within_4hrs_clean[["year"]],
    time_in_ed_clean[["year"]],
]).drop_duplicates()

In [16]:
# Sort the year and reset index
financial_year_dim = (financial_year_dim.sort_values("year").reset_index(drop=True))


In [17]:
# Create start-year and end-year column

financial_year_dim["start_year"] = (financial_year_dim["year"].str[:4].astype(int))

financial_year_dim["end_year"] = (financial_year_dim["start_year"] + 1)

In [18]:
# Rename year to financial year

financial_year_dim = financial_year_dim.rename(columns={"year": "financial_year"})

In [19]:
# Create the surrogate key
financial_year_dim.insert(0, "financial_year_key", range(1, len(financial_year_dim) + 1))


In [47]:
# Inspect it

display(financial_year_dim)

,financial_year_key,financial_year,start_year,end_year
0,1,2011–12,2011,2012
1,2,2012–13,2012,2013
2,3,2013–14,2013,2014
3,4,2014–15,2014,2015
4,5,2015–16,2015,2016
5,6,2016–17,2016,2017
6,7,2017–18,2017,2018
7,8,2018–19,2018,2019
8,9,2019–20,2019,2020
9,10,2020–21,2020,2021


In [21]:
# Validate
assert financial_year_dim["financial_year_key"].is_unique
assert financial_year_dim["financial_year"].is_unique
assert financial_year_dim["start_year"].is_unique
assert (financial_year_dim["end_year"]== financial_year_dim["start_year"] + 1).all()

In [22]:
# Create triage_category_dim

triage_category_dim = pd.concat([presentations_clean[["triage_category"]], seen_on_time_clean[["triage_category"]]]).drop_duplicates()


In [23]:
# Define the clinical severity order and order the triage category
triage_order = {
    "Resuscitation": 1,
    "Emergency": 2,
    "Urgent": 3,
    "Semi-Urgent": 4,
    "Non-Urgent": 5}

triage_category_dim["triage_order"] = (triage_category_dim["triage_category"].map(triage_order))


In [24]:
# Sort and reset the index

triage_category_dim = (triage_category_dim.sort_values("triage_order").reset_index(drop=True))


In [26]:
# Add the surrogate key

triage_category_dim.insert(0, "triage_category_key", range(1, len(triage_category_dim) + 1))

In [29]:
# Inspect the data

display(triage_category_dim)

,triage_category_key,triage_category,triage_order
0,1,Resuscitation,1
1,2,Emergency,2
2,3,Urgent,3
3,4,Semi-Urgent,4
4,5,Non-Urgent,5


In [31]:
# Validate 

assert triage_category_dim["triage_category_key"].is_unique
assert triage_category_dim["triage_category"].is_unique
assert triage_category_dim["triage_order"].notna().all()
assert triage_category_dim["triage_order"].is_unique
assert set(triage_category_dim["triage_category"]) == set(triage_order)

In [32]:
# Create patient_cohort_dim

# Collect unique cohort values

patient_cohort_dim = pd.concat([within_4hrs_clean[["patient_cohort"]], time_in_ed_clean[["patient_cohort"]]]).drop_duplicates()

In [33]:
patient_cohort_dim["patient_cohort"].value_counts()

patient_cohort
All patients                          1
Emergency                             1
Non-Urgent                            1
Not subsequently admitted patients    1
Resuscitation                         1
Semi-Urgent                           1
Subsequently admitted patients        1
Urgent                                1
Name: count, dtype: int64

In [34]:
# Create a cohort_type field

cohort_type = {
    "All patients": "Overall",
    "Resuscitation": "Triage",
    "Emergency": "Triage",
    "Urgent": "Triage",
    "Semi-Urgent": "Triage",
    "Non-Urgent": "Triage",
    "Subsequently admitted patients": "Admission status",
    "Not subsequently admitted patients": "Admission status"}


patient_cohort_dim["cohort_type"] = (
    patient_cohort_dim["patient_cohort"]
    .map(cohort_type)
)





In [35]:
# Sort the cohort into a logical order

cohort_order = {
    "All patients": 1,
    "Resuscitation": 2,
    "Emergency": 3,
    "Urgent": 4,
    "Semi-Urgent": 5,
    "Non-Urgent": 6,
    "Subsequently admitted patients": 7,
    "Not subsequently admitted patients": 8
}

# Map the order 
patient_cohort_dim["cohort_order"] = (
    patient_cohort_dim["patient_cohort"]
    .map(cohort_order)
)

In [36]:
# Sort and reset the index

patient_cohort_dim = (patient_cohort_dim.sort_values("cohort_order").reset_index(drop=True))

In [37]:
# Add the surrogate key

patient_cohort_dim.insert(0, "patient_cohort_key", range(1, len(patient_cohort_dim) + 1))

In [38]:
# Inspect the data

display(patient_cohort_dim)

,patient_cohort_key,patient_cohort,cohort_type,cohort_order
0,1,All patients,Overall,1
1,2,Resuscitation,Triage,2
2,3,Emergency,Triage,3
3,4,Urgent,Triage,4
4,5,Semi-Urgent,Triage,5
5,6,Non-Urgent,Triage,6
6,7,Subsequently admitted patients,Admission status,7
7,8,Not subsequently admitted patients,Admission status,8


In [39]:
# Validate

assert patient_cohort_dim["patient_cohort_key"].is_unique
assert patient_cohort_dim["patient_cohort"].is_unique
assert patient_cohort_dim["cohort_type"].notna().all()
assert patient_cohort_dim["cohort_order"].notna().all()
assert patient_cohort_dim["cohort_order"].is_unique
assert set(patient_cohort_dim["patient_cohort"]) == set(cohort_type)

In [40]:
# Create peer_group_dim

# Collect unique peer group values

peer_group_dim = pd.concat([
    seen_on_time_clean[["peer_group"]],
    within_4hrs_clean[["peer_group"]],
    time_in_ed_clean[["peer_group"]]
]).drop_duplicates()

In [ ]:
# Remove missing values (at National/LHN level) from the dimension table, sort and reset index

peer_group_dim = (peer_group_dim.dropna(subset=["peer_group"]).drop_duplicates().sort_values("peer_group").reset_index(drop=True))

In [42]:
# Add the surrogate key

peer_group_dim.insert(0, "peer_group_key", range(1, len(peer_group_dim) + 1))

In [ ]:
# Inspect

display(peer_group_dim)

,peer_group_key,peer_group
0,1,Children's hospitals
1,2,Large metropolitan hospitals
2,3,Large regional hospitals
3,4,Major hospitals
4,5,Medium metropolitan hospitals
5,6,Medium regional hospitals
6,7,National
7,8,Small hospitals
8,9,Unpeered


In [44]:
# Validate
assert peer_group_dim["peer_group_key"].is_unique
assert peer_group_dim["peer_group"].is_unique
assert peer_group_dim["peer_group"].notna().all()

In [46]:
# Final check of all the dimension table

print("Dimension table sizes:")
print("Reporting units:", len(reporting_unit_dim))
print("Financial years:", len(financial_year_dim))
print("Triage categories:", len(triage_category_dim))
print("Patient cohorts:", len(patient_cohort_dim))
print("Peer groups:", len(peer_group_dim))

Dimension table sizes:
Reporting units: 402
Financial years: 14
Triage categories: 5
Patient cohorts: 8
Peer groups: 9


### Create fact tables

In [49]:
# Create presentations_fact table

presentations_fact = presentations_clean.copy()


In [50]:
# Add reporting_unit_key, financial_year_key and triage_category_key to the table

presentations_fact = presentations_fact.merge(reporting_unit_dim[["reporting_unit_key", "reporting_unit", "reporting_unit_type"]],
    on=["reporting_unit", "reporting_unit_type"],
    how="left",
    validate="many_to_one")


presentations_fact = presentations_fact.merge(financial_year_dim[["financial_year_key", "financial_year"]], 
    left_on="year", 
    right_on="financial_year", how="left", validate="many_to_one")


presentations_fact = presentations_fact.merge(triage_category_dim[["triage_category_key", "triage_category"]],
    on="triage_category",
    how="left",
    validate="many_to_one")


In [51]:
# Verify that every row has a dimension key

assert presentations_fact["reporting_unit_key"].notna().all()
assert presentations_fact["financial_year_key"].notna().all()
assert presentations_fact["triage_category_key"].notna().all()


In [52]:
# Confirm that join didn't change the row count

assert len(presentations_fact) == len(presentations_clean)

print("Rows:", len(presentations_fact))



Rows: 22678


In [53]:
# Reduce it to actual fact table attributes

presentations_fact = presentations_fact[
    ["reporting_unit_key",
        "financial_year_key",
        "triage_category_key",
        "number_of_presentations",
        "small_count_flag"
    ]
]

In [54]:
# Inspect the table
presentations_fact.head(15)

,reporting_unit_key,financial_year_key,triage_category_key,number_of_presentations,small_count_flag
0,394,1,2,647788.0,False
1,394,2,2,714124.0,False
2,394,3,2,785791.0,False
3,394,4,2,843443.0,False
4,394,5,2,899906.0,False
5,394,6,2,970883.0,False
6,394,7,2,1043705.0,False
7,394,8,2,1136383.0,False
8,394,9,2,1158156.0,False
9,394,10,2,1256584.0,False


In [55]:
# Validate the uniqueness of each record 

fact_key = [
    "reporting_unit_key",
    "financial_year_key",
    "triage_category_key"
]

assert presentations_fact.duplicated(
    subset=fact_key
).sum() == 0

In [56]:
assert len(presentations_fact) == len(presentations_clean)
print(len(presentations_fact))

22678


In [57]:
# Create seen_on_time_fact table

seen_on_time_fact = seen_on_time_clean.copy()


In [58]:
# Join reporting_unit_key, reporting_unit_key, financial_year_key, triage_category_key and peer_group_key

seen_on_time_fact = seen_on_time_fact.merge(
    reporting_unit_dim[
        ["reporting_unit_key", "reporting_unit", "reporting_unit_type"]
    ],
    on=["reporting_unit", "reporting_unit_type"],
    how="left",
    validate="many_to_one"
)

seen_on_time_fact = seen_on_time_fact.merge(
    financial_year_dim[
        ["financial_year_key", "financial_year"]
    ],
    left_on="year",
    right_on="financial_year",
    how="left",
    validate="many_to_one"
)

seen_on_time_fact = seen_on_time_fact.merge(
    triage_category_dim[
        ["triage_category_key", "triage_category"]
    ],
    on="triage_category",
    how="left",
    validate="many_to_one"
)


seen_on_time_fact = seen_on_time_fact.merge(
    peer_group_dim[
        ["peer_group_key", "peer_group"]
    ],
    on="peer_group",
    how="left",
    validate="many_to_one"
)


In [ ]:
# Check the keys

assert seen_on_time_fact["reporting_unit_key"].notna().all()
assert seen_on_time_fact["financial_year_key"].notna().all()
assert seen_on_time_fact["triage_category_key"].notna().all()

In [60]:
# For peer group, check whether missing keys line up with peer_group_status == "not_applicable"

missing_peer_key = seen_on_time_fact["peer_group_key"].isna()

assert (seen_on_time_fact.loc[missing_peer_key, "peer_group_status"] == "not_applicable").all()


In [61]:
# Verify row count

assert len(seen_on_time_fact) == len(seen_on_time_clean)

print(len(seen_on_time_fact))


21972


In [62]:
# Reduce to the actual fact columns

seen_on_time_fact = seen_on_time_fact[
    [
        "reporting_unit_key",
        "financial_year_key",
        "triage_category_key",
        "peer_group_key",
        "number_of_presentations",
        "percentage_of_patients_seen_on_time",
        "peer_group_average",
        "small_count_flag",
        "seen_on_time_caution_flag",
        "seen_on_time_status",
        "peer_group_status"
    ]
]


In [63]:
# Validate the uniqueness of each record 

seen_on_time_fact_key = [
    "reporting_unit_key",
    "financial_year_key",
    "triage_category_key"
]

assert seen_on_time_fact.duplicated(
    subset=seen_on_time_fact_key
).sum() == 0

In [64]:
seen_on_time_fact.head(15)

,reporting_unit_key,financial_year_key,triage_category_key,peer_group_key,number_of_presentations,percentage_of_patients_seen_on_time,peer_group_average,small_count_flag,seen_on_time_caution_flag,seen_on_time_status,peer_group_status
0,394,1,2,NaN,644112.0,0.80,NaN,False,False,reported,not_applicable
1,394,2,2,NaN,710966.0,0.82,NaN,False,False,reported,not_applicable
2,394,3,2,NaN,781363.0,0.82,NaN,False,False,reported,not_applicable
3,394,4,2,NaN,839117.0,0.79,NaN,False,False,reported,not_applicable
4,394,5,2,NaN,895191.0,0.77,NaN,False,False,reported,not_applicable
5,394,6,2,NaN,963042.0,0.77,NaN,False,False,reported,not_applicable
6,394,7,2,NaN,1035004.0,0.76,NaN,False,False,reported,not_applicable
7,394,8,2,NaN,1126609.0,0.75,NaN,False,False,reported,not_applicable
8,394,9,2,NaN,1148190.0,0.75,NaN,False,False,reported,not_applicable
9,394,10,2,NaN,1244959.0,0.71,NaN,False,False,reported,not_applicable


In [65]:
# Create within_4_hrs_fact

within_4_hrs_fact = within_4hrs_clean.copy()

In [66]:
# Join reporting_unit_key, financial_year_key, patient_cohort_key and peer_group_key

within_4_hrs_fact = within_4_hrs_fact.merge(
    reporting_unit_dim[
        ["reporting_unit_key", "reporting_unit", "reporting_unit_type"]],
    on=["reporting_unit", "reporting_unit_type"],
    how="left",
    validate="many_to_one"
)


within_4_hrs_fact = within_4_hrs_fact.merge(
    financial_year_dim[
        ["financial_year_key", "financial_year"]
    ],
    left_on="year",
    right_on="financial_year",
    how="left",
    validate="many_to_one"
)

within_4_hrs_fact = within_4_hrs_fact.merge(
    patient_cohort_dim[
        ["patient_cohort_key", "patient_cohort"]
    ],
    on="patient_cohort",
    how="left",
    validate="many_to_one"
)


within_4_hrs_fact = within_4_hrs_fact.merge(
    peer_group_dim[
        ["peer_group_key", "peer_group"]
    ],
    on="peer_group",
    how="left",
    validate="many_to_one"
)

In [67]:
assert within_4_hrs_fact["reporting_unit_key"].notna().all()
assert within_4_hrs_fact["financial_year_key"].notna().all()
assert within_4_hrs_fact["patient_cohort_key"].notna().all()

In [68]:
missing_peer_key = within_4_hrs_fact["peer_group_key"].isna()

assert (within_4_hrs_fact.loc[
        missing_peer_key,
        "peer_group_status"]== "not_applicable"
).all()

In [69]:
assert len(within_4_hrs_fact) == len(within_4hrs_clean)
print(len(within_4_hrs_fact))

36289


In [70]:
# Keep only the fact attributes
within_4_hrs_fact = within_4_hrs_fact[
    [
        "reporting_unit_key",
        "financial_year_key",
        "patient_cohort_key",
        "peer_group_key",
        "number_of_presentations",
        "percentage_who_depart_ed_within_4_hrs",
        "peer_group_average",
        "small_count_flag",
        "within_4hrs_caution_flag",
        "within_4hrs_status",
        "peer_group_status"
    ]
]


In [71]:
# Validate

within_4_hrs_fact_key = [
    "reporting_unit_key",
    "financial_year_key",
    "patient_cohort_key"
]

assert within_4_hrs_fact.duplicated(
    subset=within_4_hrs_fact_key
).sum() == 0

In [ ]:
within_4_hrs_fact.head(15)

,reporting_unit_key,financial_year_key,patient_cohort_key,peer_group_key,number_of_presentations,percentage_who_depart_ed_within_4_hrs,peer_group_average,small_count_flag,within_4hrs_caution_flag,within_4hrs_status,peer_group_status
0,394,1,1,NaN,6547342.0,0.64,NaN,False,False,reported,not_applicable
1,394,2,1,NaN,6717067.0,0.67,NaN,False,False,reported,not_applicable
2,394,3,1,NaN,7195903.0,0.73,NaN,False,False,reported,not_applicable
3,394,4,1,NaN,7366442.0,0.73,NaN,False,False,reported,not_applicable
4,394,5,1,NaN,7465869.0,0.73,NaN,False,False,reported,not_applicable
5,394,6,1,NaN,7755606.0,0.72,NaN,False,False,reported,not_applicable
6,394,7,1,NaN,8017492.0,0.71,NaN,False,False,reported,not_applicable
7,394,8,1,NaN,8352192.0,0.70,NaN,False,False,reported,not_applicable
8,394,9,1,NaN,8236159.0,0.69,NaN,False,False,reported,not_applicable
9,394,10,1,NaN,8808357.0,0.67,NaN,False,False,reported,not_applicable


In [73]:
# Create time_in_ed_fact table

time_in_ed_fact = time_in_ed_clean.copy()

In [74]:
# Create reporting_unit_key

time_in_ed_fact = time_in_ed_fact.merge(
    reporting_unit_dim[
        ["reporting_unit_key", "reporting_unit", "reporting_unit_type"]
    ],
    on=["reporting_unit", "reporting_unit_type"],
    how="left",
    validate="many_to_one"
)

time_in_ed_fact = time_in_ed_fact.merge(
    financial_year_dim[
        ["financial_year_key", "financial_year"]
    ],
    left_on="year",
    right_on="financial_year",
    how="left",
    validate="many_to_one"
)

time_in_ed_fact = time_in_ed_fact.merge(
    patient_cohort_dim[
        ["patient_cohort_key", "patient_cohort"]
    ],
    on="patient_cohort",
    how="left",
    validate="many_to_one"
)


time_in_ed_fact = time_in_ed_fact.merge(
    peer_group_dim[
        ["peer_group_key", "peer_group"]
    ],
    on="peer_group",
    how="left",
    validate="many_to_one"
)


In [75]:
assert time_in_ed_fact["reporting_unit_key"].notna().all()
assert time_in_ed_fact["financial_year_key"].notna().all()
assert time_in_ed_fact["patient_cohort_key"].notna().all()

In [76]:
missing_peer_key = time_in_ed_fact["peer_group_key"].isna()

assert (time_in_ed_fact.loc[missing_peer_key, "peer_group_status"]== "not_applicable").all()

In [77]:
assert len(time_in_ed_fact) == len(time_in_ed_clean)
print(len(time_in_ed_fact))

13611


In [78]:
time_in_ed_fact = time_in_ed_fact[
    [
        "reporting_unit_key",
        "financial_year_key",
        "patient_cohort_key",
        "peer_group_key",
        "number_of_presentations",
        "median_time_minutes",
        "p90_time_minutes",
        "peer_group_average_90_minutes",
        "small_count_flag",
        "median_time_caution_flag",
        "p90_time_caution_flag",
        "median_time_status",
        "p90_time_status",
        "peer_group_status"
    ]
]

In [79]:
time_in_ed_fact_key = [
    "reporting_unit_key",
    "financial_year_key",
    "patient_cohort_key"
]

assert time_in_ed_fact.duplicated(
    subset=time_in_ed_fact_key
).sum() == 0

In [80]:
time_in_ed_fact.head(15)

,reporting_unit_key,financial_year_key,patient_cohort_key,peer_group_key,number_of_presentations,median_time_minutes,p90_time_minutes,peer_group_average_90_minutes,small_count_flag,median_time_caution_flag,p90_time_caution_flag,median_time_status,p90_time_status,peer_group_status
0,394,1,1,NaN,6547342.0,178.0,508.0,NaN,False,False,False,reported,reported,not_applicable
1,394,2,1,NaN,6717067.0,173.0,476.0,NaN,False,False,False,reported,reported,not_applicable
2,394,3,1,NaN,7195903.0,160.0,425.0,NaN,False,False,False,reported,reported,not_applicable
3,394,4,1,NaN,7366442.0,161.0,422.0,NaN,False,False,False,reported,reported,not_applicable
4,394,5,1,NaN,7465869.0,164.0,413.0,NaN,False,False,False,reported,reported,not_applicable
5,394,6,1,NaN,7755606.0,168.0,420.0,NaN,False,False,False,reported,reported,not_applicable
6,394,7,1,NaN,8017492.0,173.0,434.0,NaN,False,False,False,reported,reported,not_applicable
7,394,8,1,NaN,8352192.0,178.0,449.0,NaN,False,False,False,reported,reported,not_applicable
8,394,9,1,NaN,8236159.0,176.0,450.0,NaN,False,False,False,reported,reported,not_applicable
9,394,10,1,NaN,8808357.0,181.0,480.0,NaN,False,False,False,reported,reported,not_applicable


In [81]:
assert len(presentations_fact) == len(presentations_clean)
assert len(seen_on_time_fact) == len(seen_on_time_clean)
assert len(within_4_hrs_fact) == len(within_4hrs_clean)
assert len(time_in_ed_fact) == len(time_in_ed_clean)

In [82]:
assert presentations_fact[
    ["reporting_unit_key", "financial_year_key", "triage_category_key"]
].notna().all().all()

assert seen_on_time_fact[
    ["reporting_unit_key", "financial_year_key", "triage_category_key"]
].notna().all().all()

assert within_4_hrs_fact[
    ["reporting_unit_key", "financial_year_key", "patient_cohort_key"]
].notna().all().all()

assert time_in_ed_fact[
    ["reporting_unit_key", "financial_year_key", "patient_cohort_key"]
].notna().all().all()

In [83]:
for df in [
    seen_on_time_fact,
    within_4_hrs_fact,
    time_in_ed_fact
]:
    missing_peer = df["peer_group_key"].isna()

    assert (
        df.loc[missing_peer, "peer_group_status"]
        == "not_applicable"
    ).all()

In [84]:
assert presentations_fact.duplicated(
    subset=[
        "reporting_unit_key",
        "financial_year_key",
        "triage_category_key"
    ]
).sum() == 0

assert seen_on_time_fact.duplicated(
    subset=[
        "reporting_unit_key",
        "financial_year_key",
        "triage_category_key"
    ]
).sum() == 0

assert within_4_hrs_fact.duplicated(
    subset=[
        "reporting_unit_key",
        "financial_year_key",
        "patient_cohort_key"
    ]
).sum() == 0

assert time_in_ed_fact.duplicated(
    subset=[
        "reporting_unit_key",
        "financial_year_key",
        "patient_cohort_key"
    ]
).sum() == 0

In [85]:
print("Dimensions")
print("reporting_unit_dim:", reporting_unit_dim.shape)
print("financial_year_dim:", financial_year_dim.shape)
print("triage_category_dim:", triage_category_dim.shape)
print("patient_cohort_dim:", patient_cohort_dim.shape)
print("peer_group_dim:", peer_group_dim.shape)

print("\nFacts")
print("presentations_fact:", presentations_fact.shape)
print("seen_on_time_fact:", seen_on_time_fact.shape)
print("within_4_hrs_fact:", within_4_hrs_fact.shape)
print("time_in_ed_fact:", time_in_ed_fact.shape)

Dimensions
reporting_unit_dim: (402, 4)
financial_year_dim: (14, 4)
triage_category_dim: (5, 3)
patient_cohort_dim: (8, 4)
peer_group_dim: (9, 2)

Facts
presentations_fact: (22678, 5)
seen_on_time_fact: (21972, 11)
within_4_hrs_fact: (36289, 11)
time_in_ed_fact: (13611, 14)


In [ ]:
# Save all the tables

PROCESSED_DIR = Path("../data/processed") 

In [ ]:
# Dimension tables 
reporting_unit_dim.to_csv(PROCESSED_DIR / "reporting_unit_dim.csv", index=False)

financial_year_dim.to_csv(PROCESSED_DIR / "financial_year_dim.csv", index=False)

triage_category_dim.to_csv(PROCESSED_DIR / "triage_category_dim.csv", index=False)

patient_cohort_dim.to_csv(PROCESSED_DIR / "patient_cohort_dim.csv", index=False)

peer_group_dim.to_csv(PROCESSED_DIR / "peer_group_dim.csv", index=False)


# Fact tables
presentations_fact.to_csv(PROCESSED_DIR / "presentations_fact.csv", index=False)

seen_on_time_fact.to_csv(PROCESSED_DIR / "seen_on_time_fact.csv", index=False)

within_4_hrs_fact.to_csv(PROCESSED_DIR / "within_4_hrs_fact.csv", index=False)

time_in_ed_fact.to_csv(PROCESSED_DIR / "time_in_ed_fact.csv", index=False)

In [ ]:
check_reporting_unit_dim = pd.read_csv(
    PROCESSED_DIR / "reporting_unit_dim.csv")

check_time_in_ed_fact = pd.read_csv(
    PROCESSED_DIR / "time_in_ed_fact.csv")

print(check_reporting_unit_dim.shape)
print(check_time_in_ed_fact.shape)

(402, 4)
(13611, 14)


In [90]:
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

seen_on_time_fact.csv
time_in_ed_fact.csv
financial_year_dim.csv
reporting_unit_dim.csv
patient_cohort_dim.csv
presentations_fact.csv
triage_category_dim.csv
peer_group_dim.csv
within_4_hrs_fact.csv
